<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML321ENSkillsNetwork817-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Collaborative Filtering based Recommender System using K Nearest Neighbor**


Estimated time needed: **60** minutes


## **Introduction**

In this lab, the goal is to build a collaborative filtering recommender system using K nearest neighbors (KNN). The notebook uses a user-item interaction matrix (user course rating data) in the previous labs, where each row is a user, each column is a course, and each value is that user’s rating or enrollment behavior for that course.

This method uses only the past rating patterns in the user-item matrix. If two users gave similar ratings to many courses, they are treated as similar users. Then the system can use the ratings from those similar users to estimate how a target user may rate a course they have not taken yet. The same idea can also be applied from the item side: if two courses are rated similarly by many users, they are treated as similar items.

Collaborative filtering is probably the most commonly used recommendation algorithm, there are two main types of methods: 
 - **User-based** collaborative filtering is based on the user similarity or neighborhood
 - **Item-based** collaborative filtering is based on similarity among items


**Difference among recommender system**
| Method        | Uses other users? | Uses item features? | Similarity type |
| ------------- | ----------------- | ------------------- | --------------- |
| User-based CF | ✅ yes             | ❌ no                | user–user       |
| Item-based CF | ✅ yes             | ❌ no                | item–item       |
| Content-based | ❌ no              | ✅ yes               | feature–feature |

* content-based → using user profile data for feature (BoW or genre) similarity
* user-based → using user course rating data for user similarity
* item-based → using user course rating data for course similarity

User-based collaborative filtering looks for users who are similar. This is very similar to the user clustering method done in lab 6.2.3; where we employed user profiles with feature score to calculate user similarity. 
Instead of using genre feature scores, collaborative filtering uses only the course rating itself to determine if two users are similar.


#### **User-item interaction matrix**


For most collaborative filtering-based recommender systems, the main dataset format is a 2-D matrix called the user-item interaction matrix. In the matrix,  its row is labeled as the user id/index and column labelled to be the item id/index, and the element `(i, j)` represents the rating of user `i` to item `j`.  

Below is a simple example of a user-item interaction matrix:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/module_4/images/user_item_matrix.png)


#### **K-Nearest Neighbors (KNN)-based collaborative filtering**


Each row in the user-item interaction matrix shows one user’s rating history, and each column shows how different users rated one item. This matrix is usually very *sparse*, because most users interact with only a small number of courses, and each item is rated by only a small number of users.

To decide whether two users are similar, we can compare their row vectors in the matrix. Based on these similarity scores, we can find the k nearest neighbors, which are the most similar users. 

Item-based collaborative filtering follows the same idea, but looks at the matrix by columns instead of rows. In that case, we find similar items rather than similar users. If two courses are taken or rated by similar groups of users, we can treat them as similar courses and use the known ratings to predict missing ones.

If we formulate the KNN based collaborative filtering,  **the predicted rating of user $u$ to item $i$, $\hat{r}_{ui}$** is given by:


**User-based** collaborative filtering:


$$\hat{r}_{ui} = \frac{
\sum\limits_{v \in N^k_i(u)} \text{similarity}(u, v) \cdot r_{vi}}
{\sum\limits_{v \in N^k_i(u)} \text{similarity}(u, v)}$$

$\hat{r}_{ui}$: predicted rating of user $u$ on item $i$ \
$N^k_i(u)$: top $k$ neighbors of user $u$ \
${similarity}(u, v)$: how similar **user** $u$ and $v$ are \
$r_{vi}$: rating of user $v$ on item $i$ 

$$\text{predicted rating} = \text{weighted average of ratings from similar users} = \frac{\text{sum of similarity} * \text{rating}}{\text{sum of similarity}}$$

**Item-based** collaborative filtering:


Here $N^k_i(u)$ notates the nearest k neighbors of $u$.


$$\hat{r}_{ui} = \frac{
\sum\limits_{j \in N^k_u(i)} \text{similarity}(i, j) \cdot r_{uj}}
{\sum\limits_{j \in N^k_u(i)} \text{similarity}(i, j)}$$

$\hat{r}_{ui}$: predicted rating of user $u$ on item $i$ \
$N^k_u(i)$: top $k$ neighbors of item $i$ \
${similarity}(i, j)$: how similar **item** $i$ and $j$ are \
$r_{ui}$: rating of user $u$ on item $j$ 

$$\text{predicted rating} = \text{weighted average of ratings from similar items} = \frac{\text{sum of similarity} * \text{rating}}{\text{sum of similarity}}$$

Let's illustrate how the equation works using a simple example. From the above figure, suppose we want to predict the rating of `user6` to item `Machine Learning Capstone` course. After some similarity measurements, we found that k = 4 nearest neighbors: `user2, user3, user4, user5` with similarities in array ```knn_sims```:


#### **Key differences of KNN in recommender system**
The KNN method in `surprise` and `sklearn` is based on the same nearest-neighbor idea, but they are not the same tool. `surprise` KNN is designed for recommender systems and predicts missing user-item ratings directly. `sklearn` KNN is a general machine-learning method for finding neighbors, classification, or regression on feature vectors.

----


## **Objectives**


* Perform KNN-based collaborative filtering on the user-item interaction matrix


### **Prepare and setup the lab environment**


In [ ]:
import numpy as np
import math
import pandas as pd
from surprise.model_selection import GridSearchCV
from surprise import KNNBasic
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_squared_error

### **An example of calculating user-based collaborative filtering**

In [ ]:
# An example similarity array stores the similarity of user2, user3, user4, and user5 to user6
knn_sims = np.array([0.8, 0.92, 0.75, 0.83])

Also their rating on the `Machine Learning Capstone` course are:


In [ ]:
# 2.0 means audit and 3.0 means complete the course
knn_ratings = np.array([3.0, 3.0, 2.0, 3.0]) 

So the predicted rating of `user6` to item `Machine Learning Capstone` course can be calculated as:


In [ ]:
r_u6_ml =  np.dot(knn_sims, knn_ratings)/ sum(knn_sims)
r_u6_ml

If we already know the true rating to be 3.0, then we get a prediction error RMSE (Rooted Mean Squared Error) as:


In [ ]:
true_rating = 3.0
rmse = math.sqrt(true_rating - r_u6_ml) ** 2
rmse

The predicted rating is around 2.7 (close to 3.0 with RMSE 0.22), which indicates that `user6` is also likely to complete the course `Machine Learning Capstone`. As such, we may recommend it to user6 with high confidence.


### **Load and exploring dataset**


Let's first load our dataset, i.e., a **user-item (learn-course) interaction matrix**


In [ ]:
import pandas as pd

In [ ]:
#rating_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-ML0321EN-Coursera/labs/v2/module_3/ratings.csv"
rating_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/ratings.csv"
rating_df = pd.read_csv(rating_url)

In [ ]:
rating_df.head()

The dataset contains three columns, `user id` (learner), `item id`(course), and `rating`(enrollment mode). It is similar to the `test_users_df` from lab 6.2.1.

Note that this matrix is presented as the dense or vertical form, and you may convert it to a sparse matrix using `pivot` :


In [ ]:
rating_sparse_df = rating_df.pivot(index='user', columns='item', values='rating').fillna(0).reset_index().rename_axis(index=None, columns=None)
rating_sparse_df.head()

Usually, the dense format is more preferred as it saves a lot of storage and memory space. While the benefit of the sparse matrix is it is in the nature matrix format and you could apply computations such as cosine similarity directly.

* Zero indicates missing ratings, not actual zero scores.

### **Perform KNN-based collaborative filtering with two methods** 
You may choose one of the two following implementation options of KNN-based collaborative filtering. 
- The first one is to use `scikit-surprise` which is a popular and easy-to-use Python recommendation system library. 
- The second way is to implement it with standard `numpy`, `pandas`, and `sklearn`. You may need to write a lot of low-level implementation code along the way.


#### **Implementation Option 1: Use **Surprise** library (recommended)**


*Surprise* is a Python sci-kit library for recommender systems. It is simple and comprehensive to build and test different recommendation algorithms. 

First, let's install it:


In [ ]:
#!pip install scikit-surprise

Now we import required classes and methods


In [ ]:
from surprise import KNNBasic
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy

##### **Perform KNN collaborative filtering on a sample movie review dataset**

Then, let's take a look at a **code example** how easily to perform KNN collaborative filtering on a sample movie review dataset, which contains about 100k movie ratings from users.


In the Surprise package, this process is done automatically:

The default sim_options for the `KNNBasic` algorithm are
- **Step 1: build the KNN model**
```
sim_options = {
    'name': 'MSD',
    'user_based': True,
    'min_support': 1,
    'shrinkage': 100
}

model_KNN = KNNBasic(sim_options=sim_options)
```

- **Step 2: train the KNN model**

  `fit(trainset)` learns the similarities 
The model computes similarities between user pairs (or item pairs) in the training set, so you can think of it as building a user-user similarity matrix or course-course similarity matrix (as in lab 6.2.2).


**Note:** The train set is sparse, but that is normal. Surprise does not “fill in” the missing ratings before building similarity. It computes user-user or item-item similarity only from the ratings that are actually present and shared.
  

- **Step 3: Make predictions on the test set**

  `test(testset)` uses neighbors to estimate missing ratings

`test(testset)` does not care whether the pair is “missing” in some absolute sense. It simply predicts the user-course pair you pass in.

- If the pair has a real rating that was held out from training, it can be used for accuracy evaluation like RMSE.
- If the pair is unknown and not in the trainset, it can still be predicted and used for recommendation.  


In [ ]:
# Load the movielens-100k dataset (download it if needed),
data = Dataset.load_builtin('ml-100k', prompt=False)

# sample random trainset and testset
# test set is made of 25% of the ratings.
trainset, testset = train_test_split(data, test_size=.25)

# We'll use the famous KNNBasic algorithm.
algo = KNNBasic()

# Train the algorithm on the trainset, and predict ratings for the testset
algo.fit(trainset)
predictions = algo.test(testset)

# Then compute RMSE
accuracy.rmse(predictions)

As you can see, just a couple of lines and you can apply KNN collaborative filtering on the sample movie lens dataset. The main evaluation metric is `Root Mean Square Error (RMSE)` which is a very popular rating estimation error metric used in recommender systems as well as many regression model evaluations.


##### **Load the course rating dataset for the lab task**


In [ ]:
# Save the rating dataframe to a CSV file
rating_df.to_csv("course_ratings.csv", index=False)

# Read the course rating dataset with columns user item rating
reader = Reader(
    line_format='user item rating', 
    sep=',', 
    skip_lines=1, 
    rating_scale=(2, 3))

# Load the dataset from the CSV file
course_dataset = Dataset.load_from_file("course_ratings.csv", reader=reader)

We split it into trainset and testset:


In [ ]:
trainset, testset = train_test_split(course_dataset, test_size=.3)
type(trainset)

then check how many users and items we can use to fit a KNN model:


In [ ]:
print(f"Total {trainset.n_users} users and {trainset.n_items} items in the trainingset")

##### **Perform KNN-based collaborative filtering on the user-item interaction matrix**


_Fit the KNN-based collaborative filtering model using the trainset and evaluate the results using the testset:_


In [ ]:
sim_options = {
    "name": "cosine",
    "user_based": True,  # compute  similarities between items
}
algo = KNNBasic(k = 5, sim_options=sim_options)

# Train the algorithm on the trainset, and predict ratings for the testset
algo.fit(trainset)
predictions = algo.test(testset)

# Then compute RMSE
accuracy.rmse(predictions)

In [ ]:
param_grid = {
    "k": [5, 10, 20],
    "sim_options": {
        "name": ["cosine", "msd"],
        "user_based": [False],
    },
}

gs = GridSearchCV(
    KNNBasic,
    param_grid,
    measures=["rmse"],
    cv=3, 
    n_jobs=2 # use all CPUs
)

gs.fit(course_dataset)

# convert results to DataFrame
results_df = pd.DataFrame(gs.cv_results)

results_df["sim_name"] = results_df["param_sim_options"].apply(lambda x: x["name"])
results_df["user_based"] = results_df["param_sim_options"].apply(lambda x: x["user_based"])

results_df = results_df[["param_k", "sim_name", "user_based", "mean_test_rmse"]]

results_df = results_df.sort_values("mean_test_rmse")

results_df

When you set `user_based: True`, the algorithm builds a Similarity Matrix that compares every single user to every other user.

1. The Geometry of the Crash
Think of the similarity matrix as a giant square grid.
* User-Based (`True`): We have 33,900 users, the matrix is $33,900 \times 33,900$. That is 1.15 billion cells.

* Item-Based (False): We only have 127 courses, the matrix is $127 \times 127$. That is only 16 thousand cells.
  
3. The Math of RAM
Computers store these numbers (floats) using a specific amount of memory. In Python, a standard float typically uses 8 bytes.

* Item-Based Matrix: $16,129 \text{ cells} \times 8 \text{ bytes} \approx \mathbf{126 \text{ KB}}$ (Tiny! Fits on a floppy disk).

* User-Based Matrix: $1,149,210,000 \text{ cells} \times 8 \text{ bytes} \approx \mathbf{9.2 \text{ GB}}$.

User-based similarity can require several GB of RAM, while item-based similarity requires almost no memory.

If memory is insufficient, Python may raise MemoryError, or the operating system may terminate the process.

<details>
    <summary>Click here for Hints</summary>

* Create a model by calling `KNNBasic()` class. 
* Fit it with `trainset` by using `model.fit(trainset)`.  
* Record predictions to the `testset`  by using `model.test(testset).
* Compute the accuracy by using `accuracy.rmse(predictions)`


To learn more detailed usages about _Surprise_ library, visit its website from [here](https://surprise.readthedocs.io/en/stable/getting_started.html?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML321ENSkillsNetwork817-2022-01-01)


#### **Implementation Option 2: Use `numpy`, `pandas`, and `sklearn`**


If you do not prefer the one-stop Suprise solution and want more hardcore coding practices, you may implement the KNN model using `numpy`, `pandas`, and possibly `sklearn`:


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_squared_error

# --- 1. Data Preparation ---
# Split data into 70% training and 30% testing
train, test = train_test_split(rating_df, test_size=0.3, random_state=0)

# Create User-Item matrix: rows are users, columns are items
# Missing values are filled with 0 to allow for cosine similarity calculation
X = train.pivot(index="user", columns="item", values="rating").fillna(0)
global_mean = train["rating"].mean()

# --- 2. Similarity Modeling ---
# Fit KNN using Cosine Similarity. n_neighbors=6 finds the 5 closest peers + the user themselves.
knn = NearestNeighbors(n_neighbors=6, metric="cosine", algorithm="brute").fit(X)
distances, indices = knn.kneighbors(X)

# --- 3. Collaborative Filtering Prediction ---
def predict(u, i):
    """
    Predicts rating for user 'u' and item 'i' using weighted average of neighbors.
    Fallback: Returns global training mean if user/item/neighbors are unavailable.
    """
    # Cold start check: if user or item wasn't in the training set
    if u not in X.index or i not in X.columns: 
        return global_mean
    
    uidx = X.index.get_loc(u)
    
    # Extract neighbors and similarities (excluding the first index which is the user themselves)
    neigh_idx = indices[uidx, 1:]
    sims = 1 - distances[uidx, 1:] # Convert cosine distance to similarity
    
    # Retrieve neighbor ratings for the specific item from the pivoted matrix
    ratings = X.iloc[neigh_idx, X.columns.get_loc(i)]
    
    # Filter: Only consider neighbors who have actually rated this item (rating > 0)
    mask = ratings > 0
    if not mask.any(): 
        return global_mean
    
    # Calculate Similarity-Weighted Average: Σ(sim * rating) / Σ(sim)
    s, r = sims[mask], ratings[mask]
    return np.dot(s, r) / np.sum(np.abs(s))

# --- 4. Model Evaluation ---
# Generate predictions for the test set
y_pred = [predict(u, i) for u, i in zip(test["user"], test["item"])]

# Calculate and print Root Mean Squared Error (RMSE)
rmse = np.sqrt(mean_squared_error(test["rating"], y_pred))
print(f"User-Based CF RMSE: {rmse:.4f}")

## **Summary**



In this lab, we have learned and implemented KNN-based collaborative filtering. It is probably the simplest but very effective and intuitive collaborative filtering algorithm. Since it is based on KNN, it inherits the main characteristics of KNN such as memory-intensive because you need to maintain a huge similarity matrix among users or items. In the future labs, we will learn other types of collaborative filtering which do not rely on such a huge similarity matrix to make rating predictions.


## **Authors**


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/), Su Wu
